In [1]:
class DigitIterator:
    def __init__(self, number):
        self.digits = iter(str(abs(number)))

    def __iter__(self):
        return self

    def __next__(self):
        return int(next(self.digits))


# Тесты 
iterator = DigitIterator(12345)
assert list(iterator) == [1, 2, 3, 4, 5]

iterator = DigitIterator(-6789)
assert list(iterator) == [6, 7, 8, 9]


In [2]:
class FileChunkIterator:
    def __init__(self, filepath, chunk_size):
        self.filepath = filepath
        self.chunk_size = chunk_size

    def __iter__(self):
        self.file = open(self.filepath, "rb")
        return self

    def __next__(self):
        chunk = self.file.read(self.chunk_size)
        if not chunk:
            self.file.close()
            raise StopIteration
        return chunk


# Тесты 
with open("example.txt", "w") as file:
    file.write("Hello world!!")

iterator = FileChunkIterator("example.txt", 2)
assert list(iterator) == [
    b"He", b"ll", b"o ", b"wo", b"rl", b"d!", b"!"
]


In [3]:
class SubmatrixIterator:
    def __init__(self, matrix, size):
        self.matrix = matrix
        self.size = size
        self.rows = len(matrix)
        self.cols = len(matrix[0])
        self.current_row = 0
        self.current_col = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_row + self.size > self.rows:
            raise StopIteration

        submatrix = [
            row[self.current_col:self.current_col + self.size]
            for row in self.matrix[self.current_row:self.current_row + self.size]
        ]

        self.current_col += 1
        if self.current_col + self.size > self.cols:
            self.current_col = 0
            self.current_row += 1

        return submatrix


# Тесты 
matrix = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
]
iterator = SubmatrixIterator(matrix, 2)
result = list(iterator)
expected = [
    [[1, 2], [5, 6]],
    [[2, 3], [6, 7]],
    [[3, 4], [7, 8]],
    [[5, 6], [9, 10]],
    [[6, 7], [10, 11]],
    [[7, 8], [11, 12]],
    [[9, 10], [13, 14]],
    [[10, 11], [14, 15]],
    [[11, 12], [15, 16]]
]
assert result == expected


In [4]:
import os


class RecursiveFileLineIteratorNoHidden:
    def __init__(self, directory):
        self.directory = directory
        self.files = []
        self.current_file = None
        self.file_iterator = None

        for root, _, filenames in os.walk(directory):
            for filename in filenames:
                if not filename.startswith(".") and not root.startswith("."):
                    self.files.append(os.path.join(root, filename))

        self.files = iter(self.files)

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            if not self.file_iterator:
                try:
                    filepath = next(self.files)
                    self.current_file = open(filepath, "r")
                    self.file_iterator = iter(self.current_file)
                except (StopIteration, FileNotFoundError, PermissionError):
                    raise StopIteration
            try:
                return next(self.file_iterator).strip()
            except StopIteration:
                self.file_iterator = None
                self.current_file.close()
                self.current_file = None


# Тесты 
os.makedirs("test_dir/subdir", exist_ok=True)

with open("test_dir/file1.txt", "w") as file:
    file.write("File 1 Line 1\nFile 1 Line 2\n")

with open("test_dir/subdir/file2.txt", "w") as file:
    file.write("File 2 Line 1\nFile 2 Line 2\n")

iterator = RecursiveFileLineIteratorNoHidden("test_dir")
assert list(iterator) == [
    "File 1 Line 1", "File 1 Line 2", "File 2 Line 1", "File 2 Line 2"
]

os.system("rm -r test_dir")


1